# Calcul parallèle — Interprété vs compilé · **côté Python**

*Piste Python / Colab — analogue du notebook Julia `1_compilation_et_types.jl`.*

On **mesure pour de vrai** ce que le notebook Julia ne fait que citer : la boucle Python pure,
`numpy`, et un JIT (`numba`). On notera les chiffres pour les comparer en classe au `@btime` de Julia.

> Sur Colab : `Exécution ▸ Tout exécuter`. La 1ʳᵉ cellule `numba` compile (lente une fois).

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Temps par appel (s), façon %timeit : auto-échelle puis min sur `repeat` essais."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np
N = 10_000_000
v = np.random.rand(N)        # tableau numpy (contigu, typé)
v_list = v.tolist()          # la MÊME donnée en liste Python pure
len(v_list), type(v_list[0])

## 1. Des écarts de plusieurs ordres de grandeur

Le même calcul — sommer 10 millions de nombres — écrit de trois façons.

In [ ]:
def somme_boucle(x):
    s = 0.0
    for xi in x:
        s += xi
    return s

# ⚠️ boucle Python PURE sur une liste : c'est le cas lent
t_loop = best_time(lambda: somme_boucle(v_list))
print("boucle Python pure :", ms(t_loop))

In [ ]:
# numpy : la boucle se fait en C compilé, pas en Python
t_np = best_time(lambda: np.sum(v))
print("numpy.sum          :", ms(t_np))
print(f"\nnumpy est ~{t_loop/t_np:.0f}× plus rapide que la boucle Python pure")

**Ce qu'on observe.** La boucle Python pure prend ~centaines de ms à ~1 s ; `np.sum` ~quelques ms.

`numpy` est rapide parce qu'il **ne fait pas la boucle en Python** : il appelle une routine C
compilée sur un tableau contigu typé. Dès qu'on remet une **boucle Python autour des éléments**,
on repaye l'interpréteur à chaque tour.

## 2. Le *bytecode* CPython

L'analogue Python de `@code_lowered` / `@code_native` de Julia, c'est `dis` : on voit le **bytecode**
que la machine virtuelle CPython exécute dans sa boucle d'interprétation. Pas de spécialisation par
les types, pas de code natif — chaque opération passe par l'interpréteur.

In [ ]:
import dis
def f(x):
    return 2*x + 1
dis.dis(f)

Chaque `BINARY_OP` est un appel **générique** : CPython, **à l'exécution**, doit *redécouvrir*
le type de l'objet et trouver la bonne méthode (`__mul__`, `__add__`…) — à **chaque** opération, à
**chaque** tour de boucle. C'est ce **coût par opération** qui pénalise la boucle.

Julia, lui, connaît le type **dès la compilation** (au 1ᵉʳ appel avec ces types) et génère du code natif
spécialisé : `f(3.0)` et `f(3)` ne produisent pas le même code, et le test de type **disparaît** du
code chaud.

> **Distinction clé.** *Type découvert à l'**exécution*** (Python → surcoût à chaque
> opération) vs *type connu à la **compilation*** (Julia/JIT → spécialisé une fois, puis « gratuit »).
> On la retrouvera pour le **dispatch**, les **threads** et le **GPU**.

## 3. Donner un JIT à Python : `numba`

`numba.njit` compile certaines fonctions numériques en code natif — exactement la mécanique de Julia,
ajoutée *par-dessus* Python pour un sous-ensemble du langage.

In [ ]:
try:
    from numba import njit
except ImportError:
    !pip -q install numba
    from numba import njit

@njit
def somme_numba(x):
    s = 0.0
    for i in range(x.shape[0]):
        s += x[i]
    return s

somme_numba(v)                       # 1er appel : compile (lent, une fois)
t_nb = best_time(lambda: somme_numba(v))
print("boucle numba njit  :", ms(t_nb))
print(f"≈ {t_loop/t_nb:.0f}× plus rapide que la boucle Python pure")

La boucle `numba` rejoint l'ordre de grandeur de `numpy`/Julia. Leçon : « JIT » n'est pas magique,
c'est **spécialiser par les types puis compiler** — et il faut le déclencher (décorateur + 1ʳᵉ
compilation), sur un sous-ensemble de Python seulement.

## 4. Mesurer proprement

- **Piège A — la compilation.** Le 1ᵉʳ appel `numba` inclut la compilation : on l'exclut (warm-up).
- **Piège B — une seule mesure.** On répète et on prend le **minimum** (`%timeit`, ou notre `best_time`).
- **Piège C — secondes absolues.** Raisonner en **débit** : ici, octets lus / temps.

In [ ]:
octets = v.nbytes
print(f"numpy.sum : {ms(t_np)}  |  débit ≈ {octets/t_np/1e9:.1f} Go/s")
# %timeit np.sum(v)   # <- en interactif Colab, l'outil standard

## 5. Exemple conducteur : Monte-Carlo de π

In [ ]:
import random
def pi_python(n):
    c = 0
    for _ in range(n):
        x = random.random(); y = random.random()
        if x*x + y*y <= 1.0:
            c += 1
    return 4*c/n

def pi_numpy(n):
    x = np.random.rand(n); y = np.random.rand(n)
    return 4*np.count_nonzero(x*x + y*y <= 1.0)/n

@njit
def pi_numba(n):
    c = 0
    for _ in range(n):
        x = np.random.random(); y = np.random.random()
        if x*x + y*y <= 1.0:
            c += 1
    return 4*c/n

n = 5_000_000
pi_numba(10)  # warm-up
print("π python :", ms(best_time(lambda: pi_python(n))))
print("π numpy  :", ms(best_time(lambda: pi_numpy(n))))
print("π numba  :", ms(best_time(lambda: pi_numba(n))))

## Bilan — à comparer avec Julia

| Variante | Temps (à remplir) |
|---|---|
| boucle Python pure | … |
| `numpy` | … |
| `numba njit` | … |
| **Julia (boucle `@btime`)** | … |

> La boucle Python pure est ~100× plus lente que numpy/numba/Julia, pour un code *qui se ressemble
> ligne pour ligne*. La vitesse ne vient pas d'écrire « pareil » mais de **sortir de l'interpréteur**
> (C de numpy, JIT de numba) — ce que Julia fait nativement, partout.